# Seoul 2030 Mobility Commercial-Area Analysis

Google Colab + Google Drive 환경에서 실행하는 노트북입니다.

## 사전 준비 (최초 1회)

Google Drive에 아래 구조로 데이터 파일을 업로드해 두세요:

```
내 드라이브/
└── seoul_mobility/
    └── raw/
        ├── CARD_SUBWAY_MONTH_202604.csv
        ├── bus_time_station_202604.csv
        ├── seoul_admin_dong_area.zip
        ├── seoul_living_interest_groups_202512.xlsx
        └── seoul_purpose_admdong4_in_202603*.zip  (30개)
```

월말 스냅샷 파일(39개)도 같은 `raw/` 폴더에 넣으면 월별 추세 분석까지 실행됩니다.

## Step 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2. GitHub에서 코드 받기

In [ ]:
import os

REPO_URL = "https://github.com/ah1ahwon/seoul_mobility.git"
REPO_DIR = "/content/seoul_mobility"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## Step 3. 패키지 설치

In [ ]:
!pip install -q -r requirements.txt

## Step 4. 데이터 경로 설정

`DRIVE_RAW_DIR`을 Google Drive에서 raw 파일을 올린 경로로 수정하세요.

In [ ]:
import os
from pathlib import Path

# ▼ 여기를 Drive에 업로드한 실제 경로로 수정하세요
DRIVE_BASE = Path("/content/drive/MyDrive/seoul_mobility")
DRIVE_RAW_DIR = DRIVE_BASE / "raw"
DRIVE_OUTPUT_DIR = DRIVE_BASE / "output"
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 경로 확인
raw_path = Path(DRIVE_RAW_DIR)
if not raw_path.exists():
    raise FileNotFoundError(f"Drive 경로를 찾을 수 없습니다: {DRIVE_RAW_DIR}\n"
                            "Google Drive에 파일을 업로드했는지, 경로가 맞는지 확인하세요.")

files = list(raw_path.iterdir())
print(f"raw/ 파일 수: {len(files)}개")
for f in sorted(files)[:10]:
    print(" ", f.name)
if len(files) > 10:
    print(f"  ... 외 {len(files)-10}개")

# 분석 스크립트에 경로 전달
os.environ["SEOUL_RAW_DIR"] = str(DRIVE_RAW_DIR)
os.environ["SEOUL_OUTPUT_DIR"] = str(DRIVE_OUTPUT_DIR)
print("\nSEOUL_RAW_DIR 설정 완료:", DRIVE_RAW_DIR)
print("SEOUL_OUTPUT_DIR 설정 완료:", DRIVE_OUTPUT_DIR)


## Step 5. 분석 실행

In [ ]:
!python3 seoul_mobility_analysis.py

## Step 6. 결과 확인

In [ ]:
import os
import pandas as pd

OUTPUT = Path(os.environ.get("SEOUL_OUTPUT_DIR") or globals().get("DRIVE_OUTPUT_DIR", Path("/content/drive/MyDrive/seoul_mobility/output")))

# 방문성 후보 Top 20 / Bottom 5
visitor = pd.read_csv(OUTPUT / "processed/visitor_candidate_summary.csv")
print("=== 방문성 후보 Top 20 ===")
display(visitor.head(20))
print("=== 방문성 후보 Bottom 5 ===")
display(visitor.tail(5).sort_values("adjusted_mobility_score"))


In [ ]:
# 혼재형 (상권+거주) Top 20 / Bottom 5
mixed = pd.read_csv(OUTPUT / "processed/mixed_commercial_residential_summary.csv")
print("=== 혼재형 Top 20 ===")
display(mixed.head(20))
print("=== 혼재형 Bottom 5 ===")
display(mixed.tail(5).sort_values("adjusted_mobility_score"))

In [ ]:
# 월별 추세 Top 20
trend = pd.read_csv(OUTPUT / "processed/monthly_candidate_trend_summary.csv")
print("=== 월별 추세 Top 20 ===")
display(trend.head(20))

## Step 7. 결과를 Google Drive에 저장 (선택)

Colab 세션이 종료되면 `/content/` 아래 파일은 사라집니다.
결과물을 Drive에 복사해 두려면 아래 셀을 실행하세요.

In [ ]:
processed_dir = Path(DRIVE_OUTPUT_DIR) / "processed"
if not processed_dir.exists():
    raise FileNotFoundError(f"Drive 결과 폴더가 없습니다: {processed_dir}")
print("결과는 Drive에 바로 저장되어 있습니다:", DRIVE_OUTPUT_DIR)
print("processed 파일 수:", len(list(processed_dir.glob("*.csv"))))
